## 环境准备：加载 Qwen 大模型

本 Notebook 使用 **ModelScope** 加载 **Qwen2.5-7B-Instruct** 模型，
替代原有的 MockLLM / 模拟 LLM，实现真实的模型推理。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

In [ ]:
# ============================================================
# 安装依赖（如需要，取消注释后运行）
# ============================================================
# !pip install modelscope transformers torch -q

# ============================================================
# QwenLLM 封装类：基于 ModelScope 加载 Qwen2.5 模型
# ============================================================
from modelscope import AutoModelForCausalLM, AutoTokenizer
import torch


class QwenLLM:
    """
    基于 ModelScope 的 Qwen2.5 大模型封装类

    支持：
    - system prompt 设置
    - 多轮对话上下文维护
    - GPU / CPU 自动检测
    - 温度与生成长度控制
    """

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        """
        初始化 Qwen 模型

        Args:
            model_name: 模型 ID，默认 7B；低显存可改为 "Qwen/Qwen2.5-3B-Instruct"
            device: 指定设备，None 表示自动检测
        """
        # GPU / CPU 自动检测
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")
        print(f"[QwenLLM] 提示: 如显存不足，可替换为 Qwen/Qwen2.5-3B-Instruct")

        # 加载模型和分词器
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # 多轮对话历史
        self.messages = []

        print(f"[QwenLLM] 模型加载完成")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """
        对话接口

        Args:
            user_message: 用户消息
            system_prompt: 系统提示词（可选）
            max_new_tokens: 最大生成 token 数
            temperature: 采样温度

        Returns:
            模型生成的回复文本
        """
        # 构建消息列表
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        # 应用聊天模板
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # 生成回复
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # 更新对话历史
        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})

        return response

    def reset(self):
        """清空对话历史"""
        self.messages = []


# 初始化模型（首次运行需要下载，请耐心等待）
llm = QwenLLM(model_name="Qwen/Qwen2.5-7B-Instruct")
# 如显存不足，请使用：llm = QwenLLM(model_name="Qwen/Qwen2.5-3B-Instruct")

print("\n模型就绪，可以在后续 cell 中使用 llm.chat() 进行对话")

# 01 - ReAct 与 Chain-of-Thought：Agent 的推理引擎

## 学习目标

- 深入理解 Chain-of-Thought (CoT)  prompting 原理
- 掌握 ReAct (Reasoning + Acting) 模式的核心机制
- 动手实现 ReAct Agent 的完整流程
- 对比不同推理模式的适用场景

---

## 1. Chain-of-Thought (CoT) 思维链

### 1.1 什么是 CoT？

**Chain-of-Thought** 是一种提示工程技术，通过引导 LLM **显式地展示思考过程**，显著提升推理能力。

**核心思想**：让模型像人类解题一样，一步一步地思考，而不是直接给出答案。

### 1.2 标准 Prompting vs CoT Prompting

**标准 Prompting（直接提问）**：

```
Q: 一个农场有鸡和兔，头共 35 个，脚共 94 只。鸡和兔各有多少只？
A: 鸡 23 只，兔 12 只
```

**CoT Prompting（引导思考）**：

```
Q: 一个农场有鸡和兔，头共 35 个，脚共 94 只。鸡和兔各有多少只？
A: 让我一步一步思考：
   假设全是鸡，那么脚应该有 35 × 2 = 70 只
   实际多了 94 - 70 = 24 只脚
   每只兔比鸡多 2 只脚
   所以兔有 24 ÷ 2 = 12 只
   鸡有 35 - 12 = 23 只
   验证：23 × 2 + 12 × 4 = 46 + 48 = 94 ✓
   答案：鸡 23 只，兔 12 只
```

### 1.3 CoT 的变体

| 变体 | 描述 | 适用场景 |
|------|------|----------|
| **Zero-shot CoT** | 直接添加 "Let's think step by step" | 简单推理任务 |
| **Few-shot CoT** | 提供几个带推理过程的示例 | 复杂推理任务 |
| **Self-Consistency** | 多次采样，取多数答案 | 需要高准确率的任务 |
| **Tree of Thoughts** | 探索多个推理路径 | 开放性问题、创意任务 |
| **Automatic CoT** | 自动构建示例 | 减少人工编写示例的工作量 |

---

## 2. ReAct：推理与行动的协同

### 2.1 ReAct 的核心思想

**ReAct**（Reasoning + Acting）由 Google Research 于 2022 年提出，是一种将**推理（Reasoning）**和**行动（Acting）**交替进行的 Agent 设计模式。

**关键洞察**：
- 纯推理（如 CoT）缺乏与外部世界的交互
- 纯行动（如工具调用）缺乏深层次的推理规划
- **ReAct 将两者结合**，形成推理 → 行动 → 观察 → 再推理的循环

### 2.2 ReAct 的工作流程

```
用户提问："2024年诺贝尔文学奖得主是谁？他的代表作有哪些？"

Step 1 - 思考（Thought）：
    我需要查询 2024 年诺贝尔文学奖得主的信息
    → 行动（Action）：调用搜索工具
    → 观察（Observation）：获得搜索结果

Step 2 - 思考（Thought）：
    搜索结果显示得主是韩江（Han Kang）
    我还需要了解她的代表作
    → 行动（Action）：再次搜索代表作信息
    → 观察（Observation）：获得代表作列表

Step 3 - 思考（Thought）：
    现在我有足够信息回答用户了
    → 行动（Action）：生成最终答案

最终答案：2024年诺贝尔文学奖得主是韩江...
```

### 2.3 ReAct 的循环结构

```
┌─────────────────────────────────────────┐
│              用户输入                    │
└─────────────────┬───────────────────────┘
                  │
                  ▼
┌─────────────────────────────────────────┐
│  思考（Thought）                         │
│  "我需要做什么？"                        │
└─────────────────┬───────────────────────┘
                  │
                  ▼
┌─────────────────────────────────────────┐
│  行动（Action）                          │
│  调用工具 / 执行操作                     │
└─────────────────┬───────────────────────┘
                  │
                  ▼
┌─────────────────────────────────────────┐
│  观察（Observation）                     │
│  获取工具返回结果                        │
└─────────────────┬───────────────────────┘
                  │
                  │ 是否完成任务？
                  │ 是 → 生成最终答案
                  │ 否 → 继续循环
                  │
                  └──────────────→ （回到思考）
```

---

## 3. 动手实现 ReAct Agent

下面我们实现一个完整的 ReAct Agent，包含：
- 推理引擎（模拟 LLM 的思考过程）
- 工具系统（搜索、计算、问答）
- 记忆管理（维护 Thought-Action-Observation 历史）
- 循环控制（决定何时停止）

In [ ]:
# ============================================================
# ReAct Agent 完整实现（已集成 QwenLLM）
# ============================================================

# ReAct Agent 完整实现
import json
import re
from typing import Dict, List, Callable, Any

class ReActAgent:
    """
    ReAct Agent 实现
    
    核心循环：Thought → Action → Observation → ... → Answer
    """
    
    def __init__(self, max_iterations: int = 10):
        self.max_iterations = max_iterations
        self.tools: Dict[str, Dict] = {}
        self.memory: List[Dict] = []  # 存储 Thought-Action-Observation 历史
    
    def register_tool(self, name: str, func: Callable, description: str):
        """注册工具"""
        self.tools[name] = {
            "func": func,
            "description": description
        }
    
    def think(self, query: str, history: List[Dict]) -> str:
        """
        思考步骤：根据用户问题和历史记录，决定下一步行动
        
        实际应用中，这里应该调用 LLM API
        这里我们模拟 LLM 的推理过程
        """
        
        # 构建提示词（模拟）
        prompt = self._build_prompt(query, history)
        
        # 模拟 LLM 思考（实际应调用 API）
        thought = self._simulate_llm_think(prompt)
        
        return thought
    
    def _build_prompt(self, query: str, history: List[Dict]) -> str:
        """构建 ReAct 提示词"""
        
        # 工具描述
        tools_desc = "\n".join([
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        ])
        
        # 历史记录
        history_text = ""
        for step in history:
            if "thought" in step:
                history_text += f"\nThought: {step['thought']}"
            if "action" in step:
                history_text += f"\nAction: {step['action']}"
            if "observation" in step:
                history_text += f"\nObservation: {step['observation']}"
        
        prompt = f"""
回答以下问题，请使用 ReAct 模式：交替进行思考（Thought）和行动（Action）。

可用工具：
{tools_desc}

问题：{query}

{history_text}

请输出下一步的 Thought 和 Action（如果需要）：
Thought: """
        
        return prompt
    
    def _simulate_llm_think(self, prompt: str) -> str:
        """
        使用 QwenLLM 进行思考推理

        注意：以下为无模型时的备选方案（基于规则匹配的模拟实现）。
        """
        # ---- 无模型时的备选方案（已注释）----
        # query = prompt.split("问题：")[-1].split("\n")[0].strip()
        # history = self.memory
        # if "天气" in query and not any("天气" in str(h) for h in history):
        #     return "我需要查询天气信息。让我使用搜索工具获取当前天气。"
        # elif "计算" in query or "等于" in query:
        #     if not any("calculate" in str(h) for h in history):
        #         return "这是一个数学计算问题，我需要使用计算工具。"
        #     else:
        #         return "我已经获得计算结果，可以回答用户了。"
        # elif len(history) >= 2:
        #     return "我已经收集到足够的信息，可以给出最终答案了。"
        # else:
        #     return "让我搜索相关信息来回答这个问题。"
        # ---- 备选方案结束 ----

        # 使用真实 Qwen 模型进行思考
        response = llm.chat(
            prompt,
            system_prompt="你是一个 ReAct Agent 的思考模块。请根据问题和历史记录，输出你的下一步思考（Thought）。保持简洁，一行即可。",
            max_new_tokens=256,
            temperature=0.3
        )
        return response.strip()
    
    def parse_action(self, thought: str) -> tuple:
        """
        从思考中解析出行动
        返回：(action_type, action_input)
        """
        
        # 根据 thought 内容决定 action
        if "搜索" in thought or "查询" in thought:
            return ("search", "相关信息")
        elif "计算" in thought:
            # 提取计算表达式
            return ("calculate", "2+2")
        else:
            return ("finish", "")
    
    def execute_action(self, action_type: str, action_input: str) -> str:
        """执行工具"""
        if action_type in self.tools:
            try:
                result = self.tools[action_type]["func"](action_input)
                return str(result)
            except Exception as e:
                return f"工具执行错误: {str(e)}"
        elif action_type == "finish":
            return "任务完成"
        else:
            return f"未知工具: {action_type}"
    
    def run(self, query: str) -> str:
        """
        运行 ReAct 循环
        
        返回：最终答案
        """
        print(f"{'='*60}")
        print(f"🚀 开始处理查询: {query}")
        print(f"{'='*60}\n")
        
        self.memory = []  # 重置记忆
        
        for i in range(self.max_iterations):
            print(f"\n📍 第 {i+1} 轮迭代")
            print("-" * 40)
            
            # Step 1: Think
            thought = self.think(query, self.memory)
            print(f"🤔 Thought: {thought}")
            
            # Step 2: Parse Action
            action_type, action_input = self.parse_action(thought)
            
            if action_type == "finish":
                print(f"\n✅ 任务完成，生成最终答案")
                answer = self._generate_answer(query)
                print(f"\n📤 最终答案: {answer}")
                return answer
            
            print(f"🔧 Action: {action_type}({action_input})")
            
            # Step 3: Execute
            observation = self.execute_action(action_type, action_input)
            print(f"👁️ Observation: {observation}")
            
            # Step 4: Remember
            self.memory.append({
                "thought": thought,
                "action": f"{action_type}({action_input})",
                "observation": observation
            })
        
        print(f"\n⚠️ 达到最大迭代次数，强制结束")
        return self._generate_answer(query)
    
    def _generate_answer(self, query: str) -> str:
        """使用 QwenLLM 生成最终答案"""
        # ---- 无模型时的备选方案（已注释）----
        # if "天气" in query:
        #     return "根据查询结果，今天天气晴朗，气温 25°C 左右。"
        # elif "计算" in query:
        #     return "计算结果为 4。"
        # else:
        #     return "根据收集到的信息，这是您需要的答案。"
        # ---- 备选方案结束 ----

        # 使用真实 Qwen 模型生成答案
        history_text = "\n".join([
            f"Thought: {step.get('thought', '')}\nObservation: {step.get('observation', '')}"
            for step in self.memory
        ])
        prompt = f"问题：{query}\n\n推理过程：\n{history_text}\n\n请根据以上推理过程给出最终答案。"

        response = llm.chat(
            prompt,
            system_prompt="你是一个智能助手，请根据推理过程给出准确、简洁的最终答案。",
            max_new_tokens=512
        )
        return response.strip()

# 定义工具函数
def search(query: str) -> str:
    """模拟搜索工具"""
    results = {
        "天气": "今天北京天气晴朗，气温 25°C，空气质量良。",
        "相关信息": "找到 5 条相关结果，包括官方文档和教程。"
    }
    for key in results:
        if key in query:
            return results[key]
    return f"搜索 '{query}' 完成，找到相关结果。"

def calculate(expression: str) -> str:
    """计算工具"""
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except:
        return "计算失败"

# 创建 ReAct Agent
agent = ReActAgent(max_iterations=5)

# 注册工具
agent.register_tool("search", search, "搜索互联网获取信息")
agent.register_tool("calculate", calculate, "执行数学计算")

In [ ]:
# 测试 ReAct Agent
result = agent.run("今天北京的天气怎么样？")

In [ ]:
# 测试计算任务
result = agent.run("帮我计算 15 乘以 23 等于多少？")

---

## 4. CoT 与 ReAct 的对比

| 维度 | CoT (Chain-of-Thought) | ReAct (Reasoning + Acting) |
|------|------------------------|---------------------------|
| **核心机制** | 纯文本推理，展示思考过程 | 推理 + 行动交替进行 |
| **外部交互** | 无 | 可调用工具/API |
| **信息获取** | 仅依赖模型内部知识 | 可获取实时外部信息 |
| **适用任务** | 数学推理、逻辑分析 | 需要实时信息的复杂任务 |
| **复杂度** | 较低 | 较高 |
| **延迟** | 较快 | 较慢（多轮交互） |
| **准确性** | 依赖模型知识截止日期 | 可获取最新信息 |

### 选择建议

- **使用 CoT**：数学问题、逻辑推理、文本分析等不需要外部信息的任务
- **使用 ReAct**：需要实时数据、多步骤操作、与外部系统交互的任务
- **组合使用**：ReAct 的 Thought 步骤中可以使用 CoT 进行深度推理

---

## 5. 高级推理模式

### 5.1 Tree of Thoughts (ToT)

ToT 将推理过程扩展为**树形结构**，探索多个可能的思考路径。

```
                    初始问题
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
    思路 A          思路 B          思路 C
    │               │               │
    ▼               ▼               ▼
 子思路 A1       子思路 B1       子思路 C1
    │               │               │
    ▼               ▼               ▼
  评估得分        评估得分        评估得分
    │               │               │
    └───────────────┴───────────────┘
                    │
                    ▼
              选择最优路径
```

### 5.2 Self-Consistency

对同一问题进行多次 CoT 推理，选择出现频率最高的答案。

```python
# Self-Consistency 伪代码
answers = []
for _ in range(5):  # 采样 5 次
    answer = llm.generate(question, temperature=0.7)
    answers.append(answer)

# 投票选择最常见的答案
final_answer = most_common(answers)
```

### 5.3 Reflection 模式

Agent 审视自己的输出，发现错误并修正。

```
Step 1: 生成初版答案
Step 2: 自我审视 "这个答案有什么问题？"
Step 3: 发现问题并修正
Step 4: 输出最终答案
```

---

## 6. 小结

### 核心要点

1. **CoT** 让 LLM 显式思考，提升推理能力
2. **ReAct** 将推理与行动结合，实现与外部世界的交互
3. **核心循环**：Thought → Action → Observation → ... → Answer
4. **高级模式**：ToT（多路径探索）、Self-Consistency（多次采样投票）、Reflection（自我反思）

### 下一步

- [02_tool_use_and_function_calling.ipynb](02_tool_use_and_function_calling.ipynb) - 学习工具调用与 Function Calling
- [03_memory_systems.ipynb](03_memory_systems.ipynb) - 深入 Agent 记忆系统

---

## 参考资源

- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- [Chain-of-Thought Prompting Elicits Reasoning in LLMs](https://arxiv.org/abs/2201.11903)
- [Tree of Thoughts: Deliberate Problem Solving with LLMs](https://arxiv.org/abs/2305.10601)
- [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171)